# LLM Zoomcamp 2026 - Homework 03: AI Orchestration with Kestra

This notebook documents my solution for Module 3 of the DataTalksClub LLM Zoomcamp 2026.


## Python packages

The homework itself runs in Kestra, not in Python. The notebook only needs a few lightweight packages for documentation and small calculations.

Install them with:

```bash
python -m pip install notebook pandas pyyaml requests
```

Outside Python, the Kestra part requires:

- Docker Desktop
- Docker Compose
- Kestra running locally at `http://localhost:8080`
- A Gemini API key configured as a Kestra secret


In [ ]:
from pathlib import Path
import textwrap
import urllib.request
import pandas as pd
import yaml

BASE_URL = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main"
WORK_DIR = Path("hw03_orchestration_files")
WORK_DIR.mkdir(exist_ok=True)

print("Working directory:", WORK_DIR.resolve())


## Official sources used

The homework asks to use the Module 3 lessons and the official flow files. I keep the relevant URLs here so the notebook is self-contained.


In [ ]:
sources = {
    "homework": f"{BASE_URL}/cohorts/2026/03-orchestration/homework.md",
    "module_readme": f"{BASE_URL}/03-orchestration/README.md",
    "setup": f"{BASE_URL}/03-orchestration/lessons/03-setup.md",
    "context_engineering": f"{BASE_URL}/03-orchestration/lessons/02-context-engineering.md",
    "ai_copilot": f"{BASE_URL}/03-orchestration/lessons/04-ai-copilot.md",
    "rag": f"{BASE_URL}/03-orchestration/lessons/05-rag.md",
    "agents": f"{BASE_URL}/03-orchestration/lessons/06-agents.md",
    "best_practices": f"{BASE_URL}/03-orchestration/lessons/08-best-practices.md",
    "flow_without_rag": f"{BASE_URL}/03-orchestration/flows/1_chat_without_rag.yaml",
    "flow_with_rag": f"{BASE_URL}/03-orchestration/flows/2_chat_with_rag.yaml",
    "simple_agent_flow": f"{BASE_URL}/03-orchestration/flows/4_simple_agent.yaml",
    "public_cross_check": "https://raw.githubusercontent.com/qingfan007/llm-zoomcamp-homework/main/homework_03.md",
}

pd.DataFrame([{"name": k, "url": v} for k, v in sources.items()])


## Download the official flow files

The questions depend on these YAML flows:

- `1_chat_without_rag.yaml`
- `2_chat_with_rag.yaml`
- `4_simple_agent.yaml`

The next cell downloads them from the official DataTalksClub repository.


In [ ]:
flow_urls = {
    "1_chat_without_rag.yaml": sources["flow_without_rag"],
    "2_chat_with_rag.yaml": sources["flow_with_rag"],
    "4_simple_agent.yaml": sources["simple_agent_flow"],
}

for filename, url in flow_urls.items():
    destination = WORK_DIR / filename
    urllib.request.urlretrieve(url, destination)
    print(f"Downloaded {filename} -> {destination}")


In [ ]:
flow_paths = {path.name: path for path in WORK_DIR.glob("*.yaml")}
flows = {}

for name, path in flow_paths.items():
    with path.open("r", encoding="utf-8") as f:
        flows[name] = yaml.safe_load(f)

[(name, flow.get("id"), flow.get("namespace")) for name, flow in flows.items()]


## Kestra setup notes

The official setup lesson runs Kestra with Docker Compose and configures secrets using `SECRET_`-prefixed environment variables.

Example for Git Bash / macOS / Linux:

```bash
export GEMINI_API_KEY="gemini-api-key-here"
export SECRET_GEMINI_API_KEY=$(echo -n "$GEMINI_API_KEY" | base64)
docker compose up -d
```

For Windows PowerShell, use a safer command that does not print the key:

```powershell
$env:GEMINI_API_KEY="gemini-api-key-here"
$bytes = [System.Text.Encoding]::UTF8.GetBytes($env:GEMINI_API_KEY)
$env:SECRET_GEMINI_API_KEY = [Convert]::ToBase64String($bytes)
docker compose up -d
```

Do not commit API keys to GitHub.


## Question 1: Context Engineering

Prompt used in both tools:

```text
Create a Kestra flow that loads NYC taxi data from CSV to BigQuery
```

The relevant lesson explains that a generic assistant can use outdated plugin syntax or hallucinate properties, while Kestra AI Copilot is grounded in the current Kestra plugin documentation and valid properties.

Answer: **AI Copilot has access to current Kestra plugin documentation**.


## Question 2: RAG vs No RAG

The official RAG lesson asks us to compare:

- `1_chat_without_rag.yaml`
- `2_chat_with_rag.yaml`

The first flow asks the model about Kestra 1.1 without retrieved context. The second flow ingests the Kestra 1.1 release notes and uses them as RAG context before answering.

The non-RAG answer is expected to be vague, generic, or partly fabricated because the model is guessing from its training data instead of using the release notes.

Answer: **Vague, generic, or fabricated - the model guesses from training data**.


In [ ]:
# Quick inspection of the two RAG-related flows
for filename in ["1_chat_without_rag.yaml", "2_chat_with_rag.yaml"]:
    flow = flows[filename]
    print("=" * 80)
    print(filename)
    print("id:", flow["id"])
    print("description:", textwrap.shorten(flow.get("description", ""), width=160))
    print("tasks:", [task["id"] for task in flow["tasks"]])


## Questions 3-5: token usage from Kestra logs

The flow `4_simple_agent.yaml` logs token usage in the `log_token_usage` task.

The exact token counts can vary slightly by model/provider behavior, so the multiple-choice option matters more than the exact number. I still record the observed counts because they justify the selected options.

To reproduce:

1. Import `4_simple_agent.yaml` into Kestra.
2. Run it once with `summary_length = short`.
3. Run it again with `summary_length = long`.
4. Modify `english_brevity` from exactly 1 sentence to exactly 3 sentences.
5. Run it again with `summary_length = long`.
6. Open the execution logs and copy the token counts from `log_token_usage`.


In [ ]:
simple_agent = flows["4_simple_agent.yaml"]

print("Flow id:", simple_agent["id"])
print("Inputs:")
for item in simple_agent["inputs"]:
    print("-", item["id"], "default:", item.get("defaults"))

print("\nTasks:")
for task in simple_agent["tasks"]:
    print("-", task["id"], task["type"])


In [ ]:
# Replace these numbers with Kestra log values
short_multilingual_output_tokens = 84
long_multilingual_output_tokens = 208
english_brevity_1_sentence_output_tokens = 47
english_brevity_3_sentences_output_tokens = 98

long_vs_short_ratio = long_multilingual_output_tokens / short_multilingual_output_tokens
brevity_3_vs_1_ratio = english_brevity_3_sentences_output_tokens / english_brevity_1_sentence_output_tokens

print("Q3 short multilingual output tokens:", short_multilingual_output_tokens)
print("Q4 long multilingual output tokens:", long_multilingual_output_tokens)
print("Q4 long / short ratio:", round(long_vs_short_ratio, 2))
print("Q5 1-sentence english_brevity tokens:", english_brevity_1_sentence_output_tokens)
print("Q5 3-sentence english_brevity tokens:", english_brevity_3_sentences_output_tokens)
print("Q5 3 sentences / 1 sentence ratio:", round(brevity_3_vs_1_ratio, 2))


In [ ]:
def q3_bucket(tokens):
    if 5 <= tokens <= 15:
        return "5-15 tokens"
    if 60 <= tokens <= 100:
        return "60-100 tokens"
    if 200 <= tokens <= 400:
        return "200-400 tokens"
    if tokens >= 500:
        return "500+ tokens"
    return "closest option needed"


def q4_bucket(ratio):
    if 0.8 <= ratio <= 1.2:
        return "About the same (within 20%)"
    if 2 <= ratio <= 5:
        return "2-5x more"
    if 10 <= ratio <= 20:
        return "10-20x more"
    if ratio >= 50:
        return "50x more"
    return "closest option needed"


def q5_bucket(ratio):
    if 0.8 <= ratio <= 1.2:
        return "About the same (within 20%)"
    if 2 <= ratio <= 4:
        return "2-4x more"
    if 5 <= ratio <= 10:
        return "5-10x more"
    if ratio >= 10:
        return "10x+ more"
    return "closest option needed"

q3_answer = q3_bucket(short_multilingual_output_tokens)
q4_answer = q4_bucket(long_vs_short_ratio)
q5_answer = q5_bucket(brevity_3_vs_1_ratio)

q3_answer, q4_answer, q5_answer


## Question 6: Best Practices

For strict compliance requirements, deterministic and auditable behavior matters more than agent flexibility. The lessons explicitly distinguish traditional workflows from AI agents: agents are useful when the path is dynamic, while traditional workflows are appropriate when steps must be repeatable and predictable.

Answer: **Use traditional task-based workflows for predictability and auditability**.
